In [ ]:
import cv2
import os
import glob
import numpy as np

# Configuración de rutas
input_dir = "../data/bill_processing/reference_components/"
output_db_dir = "../data/database/sift_database/"
output_vis_dir = "../data/database/sift_visuals/"

# Asegurar la existencia de los directorios de salida
os.makedirs(output_db_dir, exist_ok=True)
os.makedirs(output_vis_dir, exist_ok=True)

# Se utiliza el algoritmo SIFT (Scale-Invariant Feature Transform). 
# A diferencia de ORB, donde forzamos un número fijo de características (500), SIFT detecta 
# automáticamente los puntos más estables basados en umbrales de contraste local. 
# Aunque SIFT tiene un costo computacional mayor que ORB, es altamente robusto a 
# cambios de escala y rotación.
sift = cv2.SIFT_create()

# Obtener lista de componentes
files = glob.glob(os.path.join(input_dir, "*.png"))

print(f"Iniciando extracción de características SIFT desde: {input_dir}\n")

for filepath in files:
    filename = os.path.basename(filepath)
    base_name = os.path.splitext(filename)[0]
    
    img = cv2.imread(filepath)
    if img is None:
        print(f"Error: No se pudo cargar {filename}. Omitiendo.")
        continue

    # Se utiliza la imagen en escala de grises para la detección de características porque 
    # algoritmos detectores de esquinas y bordes buscan variaciones bruscas de intensidad
    # luminosa. 
    # Las imágenes con color (RGB) triplican el costo computacional sin aportar mejoras 
    # significativas a la estructura geométrica que define al billete.    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Detección de puntos clave y cálculo de descriptores
    # kp = keypoints (puntos de interés detectados). En SIFT, esto se logra buscando extremos locales en una 
    # serie de imágenes desenfocadas progresivamente (Diferencia de Gaussianas o DoG), lo que garantiza que los puntos sean invariantes a la escala.
    # des = descriptores. A diferencia del vector binario de ORB, SIFT genera para cada punto un vector de 
    # 128 dimensiones de números de punto flotante. Este vector representa un histograma de las orientaciones 
    # de los gradientes locales, haciéndolo invariante a la rotación.
    kp, des = sift.detectAndCompute(gray, None)
    
    if des is not None:
        # Guardar descriptores en formato NumPy
        db_save_path = os.path.join(output_db_dir, f"{base_name}.npy")
        np.save(db_save_path, des)
        
        # Generar visualización de puntos clave (Keypoints)
        img_with_kp = cv2.drawKeypoints(
            img, kp, None, 
            color=(0, 255, 0), 
            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
        )
        
        vis_save_path = os.path.join(output_vis_dir, f"{base_name}_sift_keypoints.jpg")
        cv2.imwrite(vis_save_path, img_with_kp)
        
        print(f"Procesado: {filename} ({len(kp)} descriptores extraídos)")
    else:
        print(f"Aviso: No se encontraron características en {filename}.")

print("\nExtracción finalizada. Descriptores y visualizaciones guardados.")

Extracting SIFT features from components...

Extracted 53 SIFT features from norm_clean_100PesosFront_comp_0.png
Extracted 200 SIFT features from norm_clean_500PesosFront_comp_3.png
Extracted 1439 SIFT features from norm_clean_50PesosPolimeroBack_comp_5.png
Extracted 97 SIFT features from norm_clean_20PesosBack_comp_1.png
Extracted 878 SIFT features from norm_clean_20PesosPolimeroFront_comp_5.png
Extracted 852 SIFT features from norm_clean_200PesosBack_comp_4.png
Extracted 781 SIFT features from norm_clean_50PesosPolimeroFront_comp_5.png
Extracted 48 SIFT features from norm_clean_20PesosPolimeroFront_comp_2.png
Extracted 622 SIFT features from norm_clean_200PesosFront_comp_3.png
Extracted 97 SIFT features from norm_clean_50PesosPolimeroBack_comp_4.png
Extracted 21 SIFT features from norm_clean_20PesosPolimeroBack_comp_0.png
Extracted 137 SIFT features from norm_clean_20PesosPolimeroFront_comp_1.png
Extracted 96 SIFT features from norm_clean_200PesosBack_comp_2.png
Extracted 36 SIFT fea